In [ ]:
from src.io import PipelinePaths

paths = PipelinePaths.discover()
PROJECT_ROOT = paths.project_root

DATA_ROOT = PROJECT_ROOT / "data/sample"

SAMPLE_ID = "44b6_0113de3b"

In [ ]:
from pathlib import Path
import json
import pandas as pd

# ------------------------------------------------------------------
# Stage 7 output directory
# ------------------------------------------------------------------

stage7_dir = (
        DATA_ROOT
        / "processed"
        / "stage_8_track_stitching"
)

# ------------------------------------------------------------------
# Load detections
# ------------------------------------------------------------------

detections = pd.read_csv(
    stage7_dir / "detections.csv"
)

# ------------------------------------------------------------------
# Load tracks
# ------------------------------------------------------------------

tracks = pd.read_csv(
    stage7_dir / "tracks.csv"
)

# ------------------------------------------------------------------
# Load metadata
# ------------------------------------------------------------------

with open(stage7_dir / "metadata.json") as f:
    metadata = json.load(f)

print(f"Loaded {len(detections):,} detections")
print(f"Loaded {len(tracks):,} track records")
print(metadata)

In [ ]:
# ============================================================
# Build track summary
# ============================================================

# Add volume information to each tracked point
track_points = (
    tracks.merge(
        detections[
            ["frame", "cell_id", "volume_voxels"]
        ],
        left_on=["frame", "cell"],
        right_on=["frame", "cell_id"],
        how="left",
    )
)

track_summary = (
    track_points
    .sort_values(["track_id", "frame"])
    .groupby("track_id")
    .agg(
        # Lifetime
        start_frame=("frame", "first"),
        end_frame=("frame", "last"),
        length=("frame", "count"),

        # Detection IDs
        start_cell=("cell", "first"),
        end_cell=("cell", "last"),

        # Start position
        start_z=("z", "first"),
        start_y=("y", "first"),
        start_x=("x", "first"),

        # End position
        end_z=("z", "last"),
        end_y=("y", "last"),
        end_x=("x", "last"),

        # Volume
        start_volume=("volume_voxels", "first"),
        end_volume=("volume_voxels", "last"),
    )
    .reset_index()
)

track_summary.head()

In [ ]:
# ============================================================
# Build frame indices
# ============================================================

tracks_starting = {}
tracks_ending = {}

for frame, group in track_summary.groupby("start_frame"):
    tracks_starting[frame] = group.copy()

for frame, group in track_summary.groupby("end_frame"):
    tracks_ending[frame] = group.copy()

print(f"{len(tracks_starting)} start-frame groups")
print(f"{len(tracks_ending)} end-frame groups")

In [ ]:
# ============================================================
# Candidate parents
# ============================================================

last_frame = detections["frame"].max()

candidate_parents = track_summary[
    track_summary["end_frame"] < last_frame
    ].copy()

print(f"Candidate parents: {len(candidate_parents)}")

candidate_parents.head()

In [ ]:
BOUNDARY_MARGIN_Z = 3
BOUNDARY_MARGIN_Y = 10
BOUNDARY_MARGIN_X = 10

In [ ]:
def touches_boundary(row):
    return (
            row["end_z"] <= BOUNDARY_MARGIN_Z
            or row["end_z"] >= 63 - BOUNDARY_MARGIN_Z
            or row["end_y"] <= BOUNDARY_MARGIN_Y
            or row["end_y"] >= 255 - BOUNDARY_MARGIN_Y
            or row["end_x"] <= BOUNDARY_MARGIN_X
            or row["end_x"] >= 255 - BOUNDARY_MARGIN_X
    )

In [ ]:
candidate_parents = candidate_parents[
    ~candidate_parents.apply(touches_boundary, axis=1)
]

In [ ]:
print(f"Candidate parents: {len(candidate_parents)}")

candidate_parents.head()

In [ ]:
print(f"Total tracks: {len(track_summary)}")

track_summary["length"].describe()